# MC++ and PyMC++ — companion notebook to the JOSS paper

This notebook reproduces every code listing in the paper
*MC++ and PyMC++: construction, manipulation and bounding of factorable
functions*, in the order the paper presents them, with the numbers it
quotes.

- Source: <https://github.com/omega-icl/mcpp>
- Package: <https://pypi.org/project/pymcpp/>
- Documentation: <https://pymcpp.readthedocs.io>

Run the cells top to bottom. Nothing beyond `pymcpp` is needed: the wheels are
self-contained, so there is no Boost, CMake or compiler to install.

In [ ]:
%pip install -q pymcpp

import pymcpp as mc
print("pymcpp", mc.__version__)

## 1. Building the expression DAG

A DAG is hosted by an `FFGraph` object and built by writing ordinary arithmetic
on `FFVar` objects. Identical subexpressions are detected and shared as they are
created: the DAG of

$$f(x,y) = e^{xy} - 2x^2y + 3, \qquad g(x,y) = \sqrt{x^2y + 4} + e^{xy}$$

shares $xy$, $e^{xy}$ and $x^2$, and comprises 16 operations rather than 21.

A *subgraph* (`FFSubgraph`) is the list of operations (`FFOp`) for a chosen set of
dependents, in topological order — what an automatic-differentiation tool would call a tape or a codelist.
It is a first-class object that can be held, inspected and re-used, so repeated
evaluation reuses one precomputed sequence instead of re-extracting it from the
graph.

In [ ]:
DAG = mc.FFGraph()
X, Y = DAG.add_var("X"), DAG.add_var("Y")
F = mc.exp(X*Y) - 2*X**2*Y + 3
G = mc.sqrt(X**2*Y + 4) + mc.exp(X*Y)
F.set("F"); G.set("G")

SGF = DAG.subgraph([F])                               # 11 operations
print(DAG.eval(SGF, [F], [X, Y], [0.5, 1.0]))         # [4.148721270700128]
print(DAG.veval(SGF, [F], [X, Y], [[0.5, 1.], [1., 2.]]))

In [ ]:
# common-subexpression elimination, in numbers
n_F  = len(DAG.subgraph([F]))
n_G  = len(DAG.subgraph([G]))
n_FG = len(DAG.subgraph([F, G]))
print(f"F: {n_F} ops,  G: {n_G} ops,  F and G together: {n_FG} ops "
      f"(not {n_F + n_G})")

# the operation list itself
DAG.output(SGF)

## 2. Symbolic differentiation

`fdiff` (forward) and `bdiff` (reverse) return the Jacobian as a sparse triplet
of **DAG nodes** that share subexpressions with the function they differentiate.
Being expressions in the same graph, the entries can be
differentiated again for Hessians, and evaluated in any of the arithmetics
of the next section — which is how one obtains bounds on a gradient over a set.

In [ ]:
rows, cols, dF = DAG.bdiff([F], [X, Y])   # sparse Jacobian as DAG nodes
print([e.str() for e in dF])
print(DAG.eval(dF, [X, Y], [0.5, 1.0]))   # [-0.35127872929987, 0.32436063535006]

In [ ]:
# forward mode gives the same numbers
_, _, dF_fwd = DAG.fdiff([F], [X, Y])
print(DAG.eval(dF_fwd, [X, Y], [0.5, 1.0]))

## 3. Bounds and relaxations

Bounding uses the same call as point evaluation: what `eval` computes is decided
by the *type* of the values it is given, not by the function called.

All of the following describe $f$ over the host set $x \in [-1,1]$,
$y \in [0,2]$, except for the `McCormick` objects, where a point is supplied and
which carry the convex and concave relaxations of $f$ on the box evaluated at
that point — here $(0.5, 1.0)$, the second constructor argument.

In [ ]:
IX, IY = mc.Interval(-1., 1.), mc.Interval(0., 2.)
print("interval  :", DAG.eval(SGF, [F], [X, Y], [IX, IY]))

xMC = mc.McCormick(IX, 0.5)                   # range [-1,1], point x = 0.5
yMC = mc.McCormick(IY, 1.0)                   # range  [0,2], point y = 1.0
xMC.sub(2, 0); yMC.sub(2, 1)                  # seed subgradients
[FMC] = DAG.eval(SGF, [F], [X, Y], [xMC, yMC])
print("McCormick :", FMC.cv, FMC.cc, FMC.cvsub)

In [ ]:
CM = mc.SCModel(3)                            # sparse Chebyshev model, order 3
[FCM] = DAG.eval(SGF, [F], [X, Y],
                 [mc.SCVar(CM, 0, IX), mc.SCVar(CM, 1, IY)])
print("Chebyshev :", FCM.B, FCM.R)
print(FCM)

In [ ]:
SUP = mc.PWLSModel(2)                         # superposition relaxation,
[FSU] = DAG.eval(SGF, [F], [X, Y],            #   8 segments per variable
                 [mc.PWLSVar(SUP, 0, IX, 8), mc.PWLSVar(SUP, 1, IY, 8)])
print("superpos  :", FSU.l(), FSU.u())
print("  separable estimators at (0.5, 1.0):",
      FSU.uval({0: 0.5, 1: 1.0}), FSU.oval({0: 0.5, 1: 1.0}))

In [ ]:
IMG = mc.PolImg()                             # polyhedral relaxation
XP, YP = mc.PolVar(IMG, X, IX, True), mc.PolVar(IMG, Y, IY, True)
[FP] = DAG.eval([F], [X, Y], [XP, YP])
IMG.generate_cuts([FP])
print("polyhedral:", len(IMG.cuts), "cuts over", len(IMG.vars), "variables")
print(IMG)

In [ ]:
# the gradient over the host set, by evaluating the derivative nodes in
# interval arithmetic
print(DAG.eval(dF, [X, Y], [IX, IY]))

## 4. Rewriting the graph

`SLift` decomposes the dependents recursively into sparse polynomial or rational
subexpressions and transcendental ones, introducing an auxiliary variable for
each lifted term — the reformulation step that precedes tailored relaxation or
quadratic reformulation in global optimization.

In [ ]:
env = mc.SLift(DAG)
env.process([F, G])
print(env)

`FFPoly` and `FFMon` expose the sparse polynomial arithmetic directly, with
conversion between monomial and Chebyshev bases and factorisation in a chosen
variable.

In [ ]:
mc.FFPoly.options.BASIS = mc.FFPoly.options.MONOM
P = mc.FFPoly(X)**2 * mc.FFPoly(Y) - 2*mc.FFPoly(X)*mc.FFPoly(Y) + 1
print("monomial basis :", P)

P.convert(mc.FFPoly.options.CHEB)
mc.FFPoly.options.BASIS = mc.FFPoly.options.CHEB
print("Chebyshev basis:", P)
print("factored in X  :", P.factor(X))

## 5. The numbers the paper quotes

Every number the paper quotes is asserted below. If the cell raises, the paper's
listings and the installed version of `pymcpp` have diverged.

In [ ]:
tol = 1e-4
f_at = DAG.eval([F], [X, Y], [0.5, 1.0])[0]
[IF] = DAG.eval(SGF, [F], [X, Y], [IX, IY])

assert n_F == 11 and n_FG == 16 and n_F + n_G == 21
assert abs(f_at - 4.148721270700128) < 1e-12
assert abs(DAG.eval(dF, [X, Y], [0.5, 1.0])[0] - (-0.3512787292998718)) < 1e-12
assert abs(DAG.eval(dF, [X, Y], [0.5, 1.0])[1] -   0.3243606353500641)  < 1e-12
assert abs(IF.l - (-0.8647)) < tol and abs(IF.u - 10.3891) < tol
assert abs(FMC.cv - 2.0) < tol and abs(FMC.cc - 8.5756) < tol
assert FMC.cv <= f_at <= FMC.cc                      # the relaxation brackets f
assert abs(FCM.B.l - (-1.5230)) < tol and abs(FCM.B.u - 7.7258) < tol
assert abs(FSU.l() - (-0.8647)) < tol and abs(FSU.u() - 9.3891) < tol
assert abs(FSU.uval({0: .5, 1: 1.}) - 2.9699) < tol
assert abs(FSU.oval({0: .5, 1: 1.}) - 4.8069) < tol
assert len(IMG.cuts) == 23 and len(IMG.vars) == 9

print(f"f(0.5, 1) = {f_at:.4f}")
print(f"  interval enclosure      [{IF.l:7.4f}, {IF.u:7.4f}]   width {IF.u - IF.l:.2f}")
print(f"  Chebyshev model (q=3)   [{FCM.B.l:7.4f}, {FCM.B.u:7.4f}]   "
      f"width {FCM.B.u - FCM.B.l:.2f}")
print(f"  superposition bounds    [{FSU.l():7.4f}, {FSU.u():7.4f}]   "
      f"width {FSU.u() - FSU.l():.2f}")
print(f"  McCormick     at (0.5,1) [{FMC.cv:6.4f}, {FMC.cc:6.4f}]")
print(f"  superposition at (0.5,1) [{FSU.uval({0: .5, 1: 1.}):6.4f}, "
      f"{FSU.oval({0: .5, 1: 1.}):6.4f}]")
print("\nall paper values reproduced")

## 6. Three-panel 3D bounding and relaxation figure

All three panels use the *same physical $x,y$ box*, the same camera, the same $z$ scale, and the same underlying expression graph. The black mesh is the exact function $f(x,y)$; the two coloured surfaces are lower and upper values. McCormick objects are re-evaluated at each grid point while retaining the **original box**. The Chebyshev model is evaluated via its polynomial part `FCM.P({...})` plus the **same global remainder** `FCM.R`. The piecewise-linear superposition estimates are obtained from the **single** model `FSU` using `uval` and `oval`.

The process may take a few seconds. Increase `N` for higher-resolution figures; `N=31` is usually adequate for a print figure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path

N = 31                         # 31 x 31 evaluation grid
xs = np.linspace(-1.0, 1.0, N)
ys = np.linspace(0.0, 2.0, N)
XX, YY = np.meshgrid(xs, ys, indexing='xy')
ZZ = np.exp(XX * YY) - 2 * XX**2 * YY + 3

mc_lo = np.empty_like(ZZ); mc_hi = np.empty_like(ZZ)
ch_lo = np.empty_like(ZZ); ch_hi = np.empty_like(ZZ)
su_lo = np.empty_like(ZZ); su_hi = np.empty_like(ZZ)

for i, (x, y) in enumerate(zip(XX.flat, YY.flat)):
    x, y = float(x), float(y)
    point = {0:x, 1:y}

    # McCormick: propagate on the original box; change only evaluation point.
    xm, ym = mc.McCormick(IX, x), mc.McCormick(IY, y)
    xm.sub(2, 0); ym.sub(2, 1)
    [fm] = DAG.eval(SGF, [F], [X, Y], [xm, ym])
    mc_lo.flat[i], mc_hi.flat[i] = fm.cv, fm.cc

    # Sparse Chebyshev: polynomial +/- the GLOBAL remainder interval.
    poly = FCM.P(point)
    ch_lo.flat[i], ch_hi.flat[i] = poly + FCM.R.l, poly + FCM.R.u

    # Piecewise-linear superposition: same precomputed model over entire box.
    su_lo.flat[i] = FSU.uval(point)
    su_hi.flat[i] = FSU.oval(point)

# Verify correct pointwise enclosures on the plotting grid.
for name, lower, upper in (('McCormick', mc_lo, mc_hi),
                           ('Chebyshev', ch_lo, ch_hi),
                           ('Superposition', su_lo, su_hi)):
    maximum_violation = max(float(np.max(lower-ZZ)), float(np.max(ZZ-upper)))
    print(f'{name:15s}: max sampled violation = {maximum_violation:+.2e}')
    if maximum_violation > 1e-7:
        raise AssertionError(f'{name} does not enclose f at all grid nodes')

# Plot three-panel surfaces, with the exact function as a thin black wireframe.
fig = plt.figure(figsize=(15.5, 5.8), facecolor='white')
blue, orange = '#2878B5', '#E69F00'
for k, (name, lower, upper) in enumerate([
    ('(a) McCormick', mc_lo, mc_hi),
    ('(b) Sparse Chebyshev, order 3', ch_lo, ch_hi),
    ('(c) PWL superposition, 8 segments', su_lo, su_hi),
]):
    ax = fig.add_subplot(1, 3, k+1, projection='3d')
    ax.plot_surface(XX, YY, lower, color=blue, alpha=0.53,
                    linewidth=0, antialiased=True, shade=False)
    ax.plot_surface(XX, YY, upper, color=orange, alpha=0.43,
                    linewidth=0, antialiased=True, shade=False)
    ax.plot_wireframe(XX, YY, ZZ, rstride=4, cstride=4,
                      color='#202020', linewidth=0.7, alpha=0.86)
    ax.set(xlim=(-1,1), ylim=(0,2), zlim=(-2,11), xlabel='$x$', ylabel='$y$', zlabel='$f$')
    ax.view_init(elev=24, azim=-62)
    ax.set_title(name, fontsize=15, pad=4)
    # Five evenly spaced ticks across each horizontal axis.
    ax.set_xticks(np.linspace(-1.0, 1.0, 5))
    ax.set_yticks(np.linspace(0.0, 2.0, 5))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax.set_zticks([-2, 2, 6, 10])
    ax.tick_params(labelsize=11, pad=0)
    # Offset the x=1.0 and y=0.0 labels at the shared front corner.
    ax.xaxis.set_tick_params(pad=0)
    ax.yaxis.set_tick_params(pad=0)
    ax.xaxis.label.set_size(15)
    ax.yaxis.label.set_size(15)
    ax.zaxis.label.set_size(15)
    ax.set_box_aspect((1.0,1.0,0.94), zoom=0.94)

legends = [Patch(facecolor=blue, alpha=0.6, label='Pointwise underestimator'),
           Patch(facecolor=orange, alpha=0.5, label='Pointwise overestimator'),
           Line2D([0], [0], color='#202020', lw=1.3, label='Exact $f(x,y)$')]
fig.legend(handles=legends, ncol=3, frameon=False,
           loc='lower center', bbox_to_anchor=(0.5, 0.015), fontsize=12)
fig.subplots_adjust(left=0.005, right=0.995, bottom=0.12, top=0.92,
                    wspace=-0.26)